In [1]:
import polars as pl
import xgboost as xgb
import numpy as np

from src.util.constants import DATA_PATH, FIXED_XGB_PARAMETERS
from src.util.common import load_from_pickle, mean_grouped_spearman_correlation  # FIXME

We test correlation only, since old meta-model predictions are probably optimised for old target.

In [2]:
new_target = "target_ender_20"
old_target = "target_cyrusd_20"  # why not 'target_cyrus_20'?

In [3]:
# XGBoost gets stuck on the full set, so we use the selected features
feature_names = load_from_pickle(DATA_PATH / 'results/selected_features.pkl')  # FIXME
required_columns = ['era', new_target, old_target] + feature_names

df_train_list = []
df_validate_list = []
for fold in range(2):
    df_train_fold = pl.read_parquet(f"{DATA_PATH}/folds/df_train_{fold}.parquet")
    df_validate_fold = pl.read_parquet(f"{DATA_PATH}/folds/df_validate_{fold}.parquet")

    df_train_fold = df_train_fold.select(required_columns)
    df_validate_fold = df_validate_fold.select(required_columns)

    # check that targets have no nulls:
    assert df_train_fold[new_target].is_null().sum() == 0
    assert df_validate_fold[new_target].is_null().sum() == 0

    df_train_list.append(df_train_fold)
    df_validate_list.append(df_validate_fold)
    del df_train_fold, df_validate_fold

In [5]:
# these (rough) parameters performed best on the new target only on selected features - but decent place to start
params = {
    'alpha_exponent': 7,
    'colsample_bytree': 0.3,
    'lambda_exponent': 8,
    'learning_rate': 0.05,
    'max_bin': 40,
    'max_depth': 9,
    'min_child_weight_exponent': 7,
    'num_boost_round': 500,  # instead of 4000 to speed up
    'subsample': 0.7
}
params["min_child_weight"] = (10 ** params["min_child_weight_exponent"] - 1) / 10 ** 5
params["alpha"] = (10 ** params["alpha_exponent"] - 1) / 10 ** 5
params["lambda"] = (10 ** params["lambda_exponent"] - 1) / 10 ** 5
params.update(FIXED_XGB_PARAMETERS)

# extract num_boost_round (not a param in xgb.train params dict)
num_boost_round = params.pop('num_boost_round')

for key in ["min_child_weight_exponent", "alpha_exponent", "lambda_exponent"]:
    del params[key]

params = {
    **FIXED_XGB_PARAMETERS,
    **params
}

In [6]:
df_result = pl.DataFrame()

for target_train in [new_target, old_target]:
    for target_test in [new_target, old_target]:

        corr_list = []

        for fold in range(2):
            dtrain = xgb.DMatrix(
                df_train_list[fold][feature_names].to_numpy(),
                label=df_train_list[fold][target_train].to_numpy()
            )
            df_validate_subset = df_validate_list[fold].sample(fraction=0.01, seed=42)
            dval = xgb.DMatrix(
                df_validate_subset[feature_names].to_numpy(),
                label=df_validate_subset[target_test].to_numpy()
            )
            model = xgb.train(
                params=params,
                dtrain=dtrain,
                num_boost_round=num_boost_round,
                # to keep track of round
                evals=[(dval, 'tiny_eval')],
                verbose_eval=100
            )
            del dtrain, dval
            print(f"Model trained.")

            dmatrix_validate = xgb.DMatrix(df_validate_list[fold][feature_names].to_numpy())
            corr = mean_grouped_spearman_correlation(
                pl.Series(model.predict(dmatrix_validate)),
                df_validate_list[fold][target_test],
                df_validate_list[fold]['era']
            )
            corr_list.append(corr)
            del dmatrix_validate
            print(f"Train on {target_train}, evaluate on  {target_test}, fold {fold}: correlation {corr:.5f}")

        df_result = pl.concat([df_result, pl.DataFrame({'target_train': target_train, 'target_test': target_test, 'corr': np.mean(corr_list)})])


[0]	tiny_eval-rmse:0.22386
[100]	tiny_eval-rmse:0.22378
[200]	tiny_eval-rmse:0.22376
[300]	tiny_eval-rmse:0.22374
[400]	tiny_eval-rmse:0.22374
[499]	tiny_eval-rmse:0.22373
Model trained.
Train on target_ender_20, evaluate on  target_ender_20, fold 0: correlation 0.03427
[0]	tiny_eval-rmse:0.22152
[100]	tiny_eval-rmse:0.22149
[200]	tiny_eval-rmse:0.22146
[300]	tiny_eval-rmse:0.22145
[400]	tiny_eval-rmse:0.22144
[499]	tiny_eval-rmse:0.22142
Model trained.
Train on target_ender_20, evaluate on  target_ender_20, fold 1: correlation 0.02991
[0]	tiny_eval-rmse:0.22516
[100]	tiny_eval-rmse:0.22506
[200]	tiny_eval-rmse:0.22502
[300]	tiny_eval-rmse:0.22500
[400]	tiny_eval-rmse:0.22499
[499]	tiny_eval-rmse:0.22498
Model trained.
Train on target_ender_20, evaluate on  target_cyrusd_20, fold 0: correlation 0.03731
[0]	tiny_eval-rmse:0.22125
[100]	tiny_eval-rmse:0.22115
[200]	tiny_eval-rmse:0.22113
[300]	tiny_eval-rmse:0.22111
[400]	tiny_eval-rmse:0.22109
[499]	tiny_eval-rmse:0.22107
Model trained.

In [7]:
df_result

target_train,target_test,corr
str,str,f64
"""target_ender_20""","""target_ender_20""",0.032091
"""target_ender_20""","""target_cyrusd_20""",0.03692
"""target_cyrusd_20""","""target_ender_20""",0.030094
"""target_cyrusd_20""","""target_cyrusd_20""",0.040243


Fortunately, each target performs better when trained on the same target as well.